# Deterministic delegation with managed agents

A complete, deterministic course companion.

## Learning objectives

- Give managed agents narrow names, descriptions, and capabilities.
- Route distinct tasks through a manager.
- Preserve delegation events for inspection.

## Setup

The offline manager and its two workers are tiny local Python objects. They make orchestration visible without invoking a model.


For the optional live cell, install the dependency with `python -m pip install smolagents`.

## Complete implementation

Run the next cell to build the offline demonstration.

In [1]:
class ManagedAgent:
    def __init__(self, name, description, handler):
        self.name = name
        self.description = description
        self._handler = handler

    def run(self, request):
        return self._handler(request)


research_agent = ManagedAgent(
    "research",
    "Find a named historical guest in the embedded record.",
    lambda request: "Ada" if request == "first programmer" else "not found",
)
calculation_agent = ManagedAgent(
    "calculation",
    "Evaluate the course's deterministic arithmetic request.",
    lambda request: 6 * 7 if request == "six times seven" else None,
)

managed_agents = {
    research_agent.name: research_agent,
    calculation_agent.name: calculation_agent,
}
delegation_plan = [
    {"agent": "research", "request": "first programmer"},
    {"agent": "calculation", "request": "six times seven"},
]
delegation_log = []
delegated_results = {}
for assignment in delegation_plan:
    worker = managed_agents[assignment["agent"]]
    result = worker.run(assignment["request"])
    delegated_results[worker.name] = result
    delegation_log.append({"agent": worker.name, "result": result})

## Deterministic assertions

These checks make the lesson reproducible without a model or network connection.

In [2]:
assert delegated_results == {"research": "Ada", "calculation": 42}
assert [event["agent"] for event in delegation_log] == ["research", "calculation"]
assert all(agent.description for agent in managed_agents.values())

print("Offline assertions passed.")

Offline assertions passed.


## Results interpretation

The manager owns routing while each worker owns one capability. Agent descriptions are operational metadata: a model-based manager uses them to decide which worker to call, so they should be specific and non-overlapping.

## Credentials

The local delegation path needs none. The optional live manager checks `HF_TOKEN` and does not print the token.

In [3]:
import os

live_model_provider = os.getenv("LIVE_MODEL_PROVIDER", "huggingface").strip().casefold()
hf_token = os.getenv("HF_TOKEN")
if live_model_provider != "local" and not hf_token:
    print("SKIPPED: Set HF_TOKEN to run this optional smolagents example.")
else:
    try:
        from smolagents import CodeAgent
        if live_model_provider == "local":
            from smolagents import TransformersModel
        else:
            from smolagents import InferenceClientModel
    except ImportError:
        print("SKIPPED: smolagents is not installed; run `python -m pip install smolagents`.")
    else:
        if live_model_provider == "local":
            model = TransformersModel(
                model_id="Qwen/Qwen2.5-Coder-1.5B-Instruct",
                device_map="cpu",
                torch_dtype="auto",
                max_new_tokens=256,
            )
        else:
            model = InferenceClientModel(
                model_id="Qwen/Qwen2.5-Coder-32B-Instruct", token=hf_token
            )
        live_researcher = CodeAgent(
            tools=[], model=model, max_steps=2,
            name="researcher", description="Answer a focused historical research question.",
        )
        live_manager = CodeAgent(
            tools=[], model=model, max_steps=3, managed_agents=[live_researcher],
        )
        live_result = live_manager.run("Ask the researcher who is widely called the first programmer.")
        print(live_result)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Ask the researcher who is widely called the first programmer.                                                   │
│                                                                                                                 │
╰─ TransformersModel - Qwen/Qwen2.5-Coder-1.5B-Instruct ──────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  from researcher import researcher                                                                                
                                                                                                                   
  # Define the task                                                                                                
  task = "Identify the first programmer."                                                                          
  additional_args = {}                                                                                             
                                                                                                                   
  # Call the researcher function                                                                                   
  first_programmer = researcher(task=task, additional_args=additional_args)                                        
                                                                                                                   
  # Print the result                                                                                               
  print(f"The first programmer is {first_programmer}.")                                                            
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Code execution failed at line 'from researcher import researcher' due to: InterpreterError: Import from researcher 
is not allowed. Authorized imports are: ['unicodedata', 'stat', 'math', 'datetime', 'time', 'itertools', 're', 
'statistics', 'collections', 'random', 'queue']

... (6 outputs left out)

[Step 3: Duration 138.53 seconds| Input tokens: 7,836 | Output tokens: 618]

Reached max steps.

[Step 4: Duration 52.07 seconds| Input tokens: 9,501 | Output tokens: 628]

The first programmer widely called is Alan Turing.


## Completion checklist

- [ ] Defined two narrow managed agents.
- [ ] Delegated one research and one calculation request.
- [ ] Inspected the delegation log.

## Attribution

Original orchestration example informed by the official [smolagents agent reference](https://huggingface.co/docs/smolagents/reference/agents) and [multi-agent example](https://huggingface.co/docs/smolagents/examples/multiagents). No official notebook is copied. Notebook code is released under the Apache-2.0 metadata license.